# Multi-Strategy Quantitative Trading Dashboard

**Author:** Arnav Gupta  
**Date:** February 2026  
**Project:** Algorithmic Trading Strategy with Multi-Strategy Support
**Markets:** Nifty 50, BSE Sensex, S&P 500, Nasdaq 100, Dow Jones  
**Backtest Period:** Configurable (Default: 5 Years)

---
## 1. Introduction

### What is Algorithmic Trading?

Algorithmic trading refers to the use of computer programs and mathematical models to execute trades in financial markets. These algorithms follow a defined set of rules — based on timing, price, quantity, or other mathematical conditions — to place orders without human intervention.

### Available Strategies

This project implements **4 trading strategies**:

| Strategy | Description |
|----------|-------------|
| **Moving Average Crossover** | Classic Golden Cross/Death Cross using 50/200-day SMAs |
| **MA + RSI** | Combines trend (MA) with momentum confirmation (RSI > 50 for longs) |
| **RSI Only** | Mean-reversion: Buy when RSI < 30 (oversold), Sell when RSI > 70 (overbought) |
| **Buy & Hold** | Passive benchmark - always fully invested |

---
## 2. Setup & Configuration

In [69]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
import warnings
import os

warnings.filterwarnings('ignore')

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


In [70]:
# Strategy configuration
STRATEGY_CHOICE = 'Moving Average Crossover'  # Options: 'Moving Average Crossover', 'MA + RSI', 'RSI Only', 'Buy & Hold'

# Strategy parameters
SHORT_WINDOW = 50    # Short-term MA (days)
LONG_WINDOW = 200    # Long-term MA (days)
RSI_PERIOD = 14      # RSI period (for MA+RSI and RSI Only strategies)

# Backtest settings
INITIAL_CAPITAL = 100_000
USE_TRANSACTION_COST = False
TRANSACTION_COST_PCT = 0.1  # % per trade

# Date range
END_DATE = pd.Timestamp.today()
START_DATE = END_DATE - pd.DateOffset(years=5)

print(f"Strategy: {STRATEGY_CHOICE}")
print(f"Short MA: {SHORT_WINDOW} days | Long MA: {LONG_WINDOW} days | RSI: {RSI_PERIOD}")
print(f"Capital: ${INITIAL_CAPITAL:,.0f}")
print(f"Period: {START_DATE.date()} to {END_DATE.date()}")

Strategy: Moving Average Crossover
Short MA: 50 days | Long MA: 200 days | RSI: 14
Capital: $100,000
Period: 2021-03-10 to 2026-03-10


---
## 3. Market Data Configuration

Select from major Indian and US indices:

In [71]:
# Market/Index configuration
# Uncomment one of the following options:

# Option 1: Indian Markets (NSE)
MARKET = 'NSE'
TICKER = 'RELIANCE.NS'  # Options: RELIANCE.NS, TCS.NS, INFY.NS, HDFCBANK.NS, etc.

# Option 2: US Markets (uncomment to use)
# MARKET = 'US'
# TICKER = 'AAPL'  # Options: AAPL, MSFT, GOOGL, AMZN, NVDA, etc.

# Option 3: Indices (uncomment to use)
# MARKET = 'INDEX'
# TICKER = '^NSEI'  # Nifty 50 Index
# TICKER = '^GSPC'  # S&P 500
# TICKER = '^DJI'   # Dow Jones

print(f"Market: {MARKET}")
print(f"Ticker: {TICKER}")

Market: NSE
Ticker: RELIANCE.NS


---
## 4. Data Collection

Download historical OHLCV data from Yahoo Finance:

In [72]:
def download_data(ticker, start, end):
    """Download OHLC data from Yahoo Finance."""
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if raw.empty:
        return pd.DataFrame()
    df = raw[['Close']].copy()
    # Handle MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]
    df.dropna(inplace=True)
    return df

# Download data
df = download_data(TICKER, START_DATE, END_DATE)

print(f"✅ Downloaded {len(df)} trading days of data for {TICKER}")
print(f"📅 Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"\n--- Sample Data ---")
df.head(10)

✅ Downloaded 1236 trading days of data for RELIANCE.NS
📅 Date range: 2021-03-10 to 2026-03-10

--- Sample Data ---


,Close
Date,
2021-03-10,989.364258
2021-03-12,969.254700
2021-03-15,956.241150
2021-03-16,952.477661
2021-03-17,931.959961
2021-03-18,910.988831
2021-03-19,944.043884
2021-03-22,935.156616
2021-03-23,946.537781


In [73]:
# Summary statistics
print("=== Summary Statistics ===")
df.describe().round(2)

=== Summary Statistics ===


,Close
count,1236.00
mean,1240.46
std,173.17
min,862.04
25%,1116.91
50%,1211.46
75%,1404.93
max,1592.30


---
## 5. Technical Indicators

In [74]:
def calculate_rsi(close, period=14):
    """Calculate Relative Strength Index."""
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    rs = np.where(avg_loss == 0, np.inf, avg_gain / avg_loss)
    rsi = 100 - (100 / (1 + rs))
    return pd.Series(rsi, index=close.index)

def calculate_sma(close, window):
    """Calculate Simple Moving Average."""
    return close.rolling(window=window, min_periods=window).mean()

print("✅ Technical indicator functions defined")

✅ Technical indicator functions defined


---
## 6. Strategy Implementation

### Strategy Logic

| Strategy | Buy Condition | Sell Condition |
|----------|---------------|----------------|
| **MA Crossover** | Short MA > Long MA (Golden Cross) | Short MA < Long MA (Death Cross) |
| **MA + RSI** | Short MA > Long MA AND RSI > 50 | Short MA < Long MA AND RSI < 50 |
| **RSI Only** | RSI < 30 (oversold) | RSI > 70 (overbought) |
| **Buy & Hold** | Always long | Never sell |

In [75]:
def apply_strategy(df, strategy_type, params):
    """
    Apply the selected strategy and generate signals.
    Returns DataFrame with Signal, Position, and strategy-specific columns.
    """
    out = df[['Close']].copy()
    
    if strategy_type == 'Moving Average Crossover':
        short_w = params['short_window']
        long_w = params['long_window']
        
        out['SMA_Short'] = calculate_sma(out['Close'], short_w)
        out['SMA_Long'] = calculate_sma(out['Close'], long_w)
        
        out['Signal'] = 0
        out.loc[out['SMA_Short'] > out['SMA_Long'], 'Signal'] = 1
        out.loc[out['SMA_Short'] < out['SMA_Long'], 'Signal'] = -1
        
    elif strategy_type == 'MA + RSI':
        short_w = params['short_window']
        long_w = params['long_window']
        rsi_p = params['rsi_period']
        
        out['SMA_Short'] = calculate_sma(out['Close'], short_w)
        out['SMA_Long'] = calculate_sma(out['Close'], long_w)
        out['RSI'] = calculate_rsi(out['Close'], rsi_p)
        
        out['Signal'] = 0
        buy_cond = (out['SMA_Short'] > out['SMA_Long']) & (out['RSI'] > 50)
        sell_cond = (out['SMA_Short'] < out['SMA_Long']) & (out['RSI'] < 50)
        out.loc[buy_cond, 'Signal'] = 1
        out.loc[sell_cond, 'Signal'] = -1
        out['Signal'] = out['Signal'].replace(0, np.nan).ffill().fillna(0).astype(int)
        
    elif strategy_type == 'RSI Only':
        rsi_p = params['rsi_period']
        
        out['RSI'] = calculate_rsi(out['Close'], rsi_p)
        
        out['Signal'] = 0
        out.loc[out['RSI'] < 30, 'Signal'] = 1   # Oversold -> Buy
        out.loc[out['RSI'] > 70, 'Signal'] = -1  # Overbought -> Sell
        out['Signal'] = out['Signal'].replace(0, np.nan).ffill().fillna(0).astype(int)
        
    elif strategy_type == 'Buy & Hold':
        out['Signal'] = 1  # Always long
        
    else:
        raise ValueError(f"Unknown strategy: {strategy_type}")
    
    # Crossover detection and position (shifted to avoid look-ahead bias)
    if strategy_type != 'Buy & Hold':
        out['Crossover'] = out['Signal'].diff()
        out['Position'] = out['Signal'].shift(1)
        out['Buy_Signal'] = out['Crossover'] == 2
        out['Sell_Signal'] = out['Crossover'] == -2
    else:
        out['Position'] = 1
        out['Crossover'] = 0
        out['Buy_Signal'] = False
        out['Sell_Signal'] = False
    
    return out

# Strategy parameters dictionary
params = {
    'short_window': SHORT_WINDOW,
    'long_window': LONG_WINDOW,
    'rsi_period': RSI_PERIOD
}

# Apply strategy
df_strat = apply_strategy(df, STRATEGY_CHOICE, params)

print(f"✅ Applied strategy: {STRATEGY_CHOICE}")
print(f"\n--- Sample with Signals ---")
df_strat.dropna().head(10)

✅ Applied strategy: Moving Average Crossover

--- Sample with Signals ---


,Close,SMA_Short,SMA_Long,Signal,Crossover,Position,Buy_Signal,Sell_Signal
Date,,,,,,,,
2021-12-30,1073.418945,1120.881179,1019.701437,1,1.0,0.0,False,False
2021-12-31,1077.536743,1117.857625,1020.142299,1,0.0,1.0,False,False
2022-01-03,1093.780640,1115.867859,1020.764929,1,0.0,1.0,False,False
2022-01-04,1118.465088,1114.327188,1021.576049,1,0.0,1.0,False,False
2022-01-05,1123.697632,1113.124136,1022.432148,1,0.0,1.0,False,False
2022-01-06,1099.536499,1110.898672,1023.270031,1,0.0,1.0,False,False
2022-01-07,1108.409180,1109.156882,1024.257133,1,0.0,1.0,False,False
2022-01-10,1109.319336,1107.695388,1025.083510,1,0.0,1.0,False,False
2022-01-11,1117.304810,1106.960999,1025.994251,1,0.0,1.0,False,False


### Trading Signals Summary

In [76]:
buy_signals = df_strat[df_strat.get('Buy_Signal', False) == True]
sell_signals = df_strat[df_strat.get('Sell_Signal', False) == True]

print(f"📈 Buy Signals (Golden Cross): {len(buy_signals)}")
print(f"📉 Sell Signals (Death Cross): {len(sell_signals)}")

if len(buy_signals) > 0:
    print(f"\n--- Buy Signal Dates ---")
    for idx in buy_signals.index:
        print(f"  {idx.date()}  |  Close: ${df_strat.loc[idx, 'Close']:.2f}")

if len(sell_signals) > 0:
    print(f"\n--- Sell Signal Dates ---")
    for idx in sell_signals.index:
        print(f"  {idx.date()}  |  Close: ${df_strat.loc[idx, 'Close']:.2f}")

📈 Buy Signals (Golden Cross): 3
📉 Sell Signals (Death Cross): 4

--- Buy Signal Dates ---
  2022-12-12  |  Close: $1192.87
  2023-06-26  |  Close: $1139.21
  2025-05-29  |  Close: $1412.16

--- Sell Signal Dates ---
  2022-10-27  |  Close: $1118.94
  2023-02-03  |  Close: $1063.18
  2024-10-23  |  Close: $1333.20
  2026-03-09  |  Close: $1424.00


---
## 7. Backtesting Engine

### Backtest Methodology

- **Initial Capital:** $100,000 (configurable)
- **Position:** Shifted by 1 day to prevent look-ahead bias
- **Transaction Costs:** Optional (configurable % per trade)
- **Benchmark:** Buy & Hold comparison

In [77]:
def backtest_strategy(df, initial_capital, transaction_cost_pct=0.0, use_transaction_cost=False):
    """
    Run backtest: strategy return = daily return * position.
    """
    out = df[['Close', 'Position']].copy()
    out['Daily_Return'] = out['Close'].pct_change()
    
    # Strategy return: only earn when position == 1 (long)
    position_active = (out['Position'] == 1).astype(int)
    out['Strategy_Return'] = out['Daily_Return'] * position_active
    out['BuyHold_Return'] = out['Daily_Return']
    
    # Apply transaction costs on position changes
    if use_transaction_cost and transaction_cost_pct != 0:
        cost_decimal = transaction_cost_pct / 100.0
        position_changes = out['Position'].diff().abs()
        trade_cost = position_changes * cost_decimal
        out['Strategy_Return'] = out['Strategy_Return'] - trade_cost
    
    # Cumulative returns and portfolio value
    out['Strategy_Cumulative'] = (1 + out['Strategy_Return']).cumprod()
    out['BuyHold_Cumulative'] = (1 + out['BuyHold_Return']).cumprod()
    out['Strategy_Value'] = initial_capital * out['Strategy_Cumulative']
    out['BuyHold_Value'] = initial_capital * out['BuyHold_Cumulative']
    
    return out.dropna(subset=['Strategy_Return'])

# Run backtest
backtest = backtest_strategy(
    df_strat, 
    INITIAL_CAPITAL,
    transaction_cost_pct=TRANSACTION_COST_PCT,
    use_transaction_cost=USE_TRANSACTION_COST
)

print(f"✅ Backtest complete")
print(f"📅 Period: {backtest.index.min().date()} to {backtest.index.max().date()}")
print(f"📊 Trading days: {len(backtest)}")
print(f"\n💰 Final Strategy Portfolio Value:   ${backtest['Strategy_Value'].iloc[-1]:,.2f}")
print(f"💰 Final Buy-and-Hold Portfolio Value: ${backtest['BuyHold_Value'].iloc[-1]:,.2f}")

✅ Backtest complete
📅 Period: 2021-03-12 to 2026-03-10
📊 Trading days: 1235

💰 Final Strategy Portfolio Value:   $109,640.07
💰 Final Buy-and-Hold Portfolio Value: $142,394.48


---
## 8. Performance Metrics

| Metric | Description |
|--------|-------------|
| **Total Return** | Percentage gain/loss over the entire period |
| **Annualized Return** | Geometric average yearly return |
| **Annualized Volatility** | Standard deviation of returns, annualized |
| **Sharpe Ratio** | Risk-adjusted return (excess return / volatility) |
| **Maximum Drawdown** | Largest peak-to-trough decline in portfolio value |

In [78]:
TRADING_DAYS_PER_YEAR = 252
RISK_FREE_RATE = 0.0

def calculate_metrics(returns, label='Strategy'):
    """Compute institutional-grade performance metrics."""
    total_ret = (1 + returns).prod() - 1
    n_years = len(returns) / TRADING_DAYS_PER_YEAR
    ann_ret = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else 0
    ann_vol = returns.std() * np.sqrt(TRADING_DAYS_PER_YEAR)
    sharpe = (ann_ret / ann_vol) if ann_vol != 0 else 0.0
    
    cum = (1 + returns).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax()
    max_dd = dd.min()
    
    return {
        'Label': label,
        'Total Return': total_ret,
        'Annualized Return': ann_ret,
        'Annualized Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
    }

def get_win_rate(backtest_df):
    """Calculate win rate: % of invested days with positive return."""
    rets = backtest_df['Strategy_Return']
    invested = rets[rets != 0]
    if len(invested) == 0:
        return 0.0
    return (invested > 0).sum() / len(invested) * 100

# Calculate metrics
strat_metrics = calculate_metrics(backtest['Strategy_Return'], STRATEGY_CHOICE)
bh_metrics = calculate_metrics(backtest['BuyHold_Return'], 'Buy & Hold')
win_rate = get_win_rate(backtest)

# Display metrics
print("=" * 70)
print("                    PERFORMANCE COMPARISON")
print("=" * 70)
print(f"\n{'Metric':<30} {'Strategy':>18} {'Buy & Hold':>18}")
print("-" * 70)
print(f"{'Total Return':<30} {strat_metrics['Total Return']*100:>17.2f}% {bh_metrics['Total Return']*100:>17.2f}%")
print(f"{'Annualized Return':<30} {strat_metrics['Annualized Return']*100:>17.2f}% {bh_metrics['Annualized Return']*100:>17.2f}%")
print(f"{'Annualized Volatility':<30} {strat_metrics['Annualized Volatility']*100:>17.2f}% {bh_metrics['Annualized Volatility']*100:>17.2f}%")
print(f"{'Sharpe Ratio':<30} {strat_metrics['Sharpe Ratio']:>18.3f} {bh_metrics['Sharpe Ratio']:>18.3f}")
print(f"{'Maximum Drawdown':<30} {strat_metrics['Max Drawdown']*100:>17.2f}% {bh_metrics['Max Drawdown']*100:>17.2f}%")
print("-" * 70)
print(f"{'Win Rate':<30} {win_rate:>17.2f}% {'N/A':>18}")
print("=" * 70)

                    PERFORMANCE COMPARISON

Metric                                   Strategy         Buy & Hold
----------------------------------------------------------------------
Total Return                                9.64%             42.39%
Annualized Return                           1.90%              7.48%
Annualized Volatility                      17.50%             22.38%
Sharpe Ratio                                0.108              0.334
Maximum Drawdown                          -24.60%            -27.18%
----------------------------------------------------------------------
Win Rate                                   50.92%                N/A


---
## 9. Interactive Visualizations (Plotly)

### 9.1 Price Chart with Signals

In [79]:
# Price chart with moving averages and signals
valid = df_strat.dropna(subset=['Close'])

fig1 = go.Figure()

# Close price
fig1.add_trace(go.Scatter(
    x=valid.index, y=valid['Close'],
    name='Close Price',
    line=dict(color='#b0bec5', width=1.5)
))

# Short MA
if 'SMA_Short' in valid.columns:
    v_ma = valid.dropna(subset=['SMA_Short'])
    fig1.add_trace(go.Scatter(
        x=v_ma.index, y=v_ma['SMA_Short'],
        name=f'{SHORT_WINDOW}-day SMA',
        line=dict(color='#2196F3', width=2)
    ))

# Long MA
if 'SMA_Long' in valid.columns:
    v_ma = valid.dropna(subset=['SMA_Long'])
    fig1.add_trace(go.Scatter(
        x=v_ma.index, y=v_ma['SMA_Long'],
        name=f'{LONG_WINDOW}-day SMA',
        line=dict(color='#FF9800', width=2)
    ))

# Buy signals
if 'Buy_Signal' in valid.columns and valid['Buy_Signal'].any():
    buys = valid[valid['Buy_Signal']]
    fig1.add_trace(go.Scatter(
        x=buys.index, y=buys['Close'],
        mode='markers',
        name='Buy Signal',
        marker=dict(
            symbol='triangle-up',
            size=14,
            color='#4CAF50',
            line=dict(width=1, color='white')
        )
    ))

# Sell signals
if 'Sell_Signal' in valid.columns and valid['Sell_Signal'].any():
    sells = valid[valid['Sell_Signal']]
    fig1.add_trace(go.Scatter(
        x=sells.index, y=sells['Close'],
        mode='markers',
        name='Sell Signal',
        marker=dict(
            symbol='triangle-down',
            size=14,
            color='#F44336',
            line=dict(width=1, color='white')
        )
    ))

fig1.update_layout(
    title=f'{TICKER} — Price Chart with {STRATEGY_CHOICE} Signals',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    template='plotly_dark',
    height=500,
    hovermode='x unified',
    legend=dict(orientation='h', y=1.02)
)

fig1.show()

### 9.2 Equity Curve

In [80]:
# Equity curve comparison
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=backtest.index, y=backtest['Strategy_Value'],
    name=STRATEGY_CHOICE,
    line=dict(color='#2196F3', width=2.5),
    fill='tozeroy',
    fillcolor='rgba(33, 150, 243, 0.15)'
))

fig2.add_trace(go.Scatter(
    x=backtest.index, y=backtest['BuyHold_Value'],
    name='Buy & Hold',
    line=dict(color='#FF9800', width=2.5)
))

fig2.add_hline(
    y=INITIAL_CAPITAL,
    line_dash='dash',
    line_color='gray',
    annotation_text='Initial Capital'
)

fig2.update_layout(
    title=f'Equity Curve — {STRATEGY_CHOICE} vs Buy & Hold',
    xaxis_title='Date',
    yaxis_title='Portfolio Value ($)',
    template='plotly_dark',
    height=450,
    hovermode='x unified',
    legend=dict(orientation='h', y=1.02)
)

fig2.show()

### 9.3 Drawdown Chart

In [81]:
# Drawdown chart
cum = (1 + backtest['Strategy_Return']).cumprod()
running_max = cum.cummax()
drawdown = (cum - running_max) / running_max

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=drawdown.index, y=drawdown * 100,
    name='Drawdown',
    line=dict(color='#F44336', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(244, 67, 54, 0.25)'
))

fig3.update_layout(
    title=f'{STRATEGY_CHOICE} — Drawdown Over Time',
    xaxis_title='Date',
    yaxis_title='Drawdown (%)',
    template='plotly_dark',
    height=400,
    hovermode='x unified'
)

fig3.show()

print(f"\n📉 Maximum Drawdown: {drawdown.min()*100:.2f}%")
print(f"📅 Drawdown occurred at: {drawdown.idxmin().date()}")


📉 Maximum Drawdown: -24.60%
📅 Drawdown occurred at: 2023-10-26


### 9.4 Daily Returns Distribution

In [82]:
# Returns distribution histogram
fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(STRATEGY_CHOICE, 'Buy & Hold')
)

fig4.add_trace(
    go.Histogram(
        x=backtest['Strategy_Return'] * 100,
        nbinsx=50,
        name='Strategy',
        marker_color='#2196F3',
        opacity=0.75
    ),
    row=1, col=1
)

fig4.add_trace(
    go.Histogram(
        x=backtest['BuyHold_Return'] * 100,
        nbinsx=50,
        name='Buy & Hold',
        marker_color='#FF9800',
        opacity=0.75
    ),
    row=1, col=2
)

fig4.update_layout(
    title='Daily Returns Distribution (%)',
    template='plotly_dark',
    height=400,
    showlegend=False
)

fig4.show()

### 9.5 Monthly Returns Heatmap

In [83]:
# Monthly returns heatmap
rets = backtest['Strategy_Return'].copy()
rets.index = pd.to_datetime(rets.index)
monthly = rets.resample('M').apply(lambda x: (1 + x).prod() - 1) * 100

years = monthly.index.year.unique()
months = list(range(1, 13))
matrix = np.full((len(years), 12), np.nan)

for i, y in enumerate(years):
    for m in months:
        sel = monthly[(monthly.index.year == y) & (monthly.index.month == m)]
        if len(sel) > 0:
            matrix[i, m - 1] = sel.iloc[0]

fig5 = go.Figure(data=go.Heatmap(
    z=matrix,
    x=[f'{m}' for m in months],
    y=[str(y) for y in years],
    colorscale='RdYlGn',
    zmid=0,
    colorbar=dict(title='Return %')
))

fig5.update_layout(
    title=f'{STRATEGY_CHOICE} — Monthly Returns (%)',
    xaxis_title='Month',
    yaxis_title='Year',
    template='plotly_dark',
    height=400
)

fig5.show()

---
## 10. Parameter Optimization

### Strategy Parameter Heatmap

Find the optimal Short MA and Long MA combinations by testing various parameters:

In [84]:
def compute_sharpe_grid(ticker_str, start_str, end_str, short_list, long_list, capital):
    """Compute Sharpe ratio for each (short, long) MA pair."""
    raw = download_data(ticker_str, start_str, end_str)
    if raw.empty:
        return np.full((len(short_list), len(long_list)), np.nan)
    
    results = []
    for sw in short_list:
        row = []
        for lw in long_list:
            if sw >= lw:
                row.append(np.nan)
                continue
            params = {'short_window': sw, 'long_window': lw, 'rsi_period': 14}
            df_s = apply_strategy(raw, 'Moving Average Crossover', params)
            bt = backtest_strategy(df_s, capital, transaction_cost_pct=0.0, use_transaction_cost=False)
            if bt.empty:
                row.append(np.nan)
                continue
            m = calculate_metrics(bt['Strategy_Return'], '')
            row.append(m['Sharpe Ratio'])
        results.append(row)
    return np.array(results)

# Parameter ranges
short_range = range(20, 81, 5)   # Short MA: 20 to 80
long_range = range(100, 301, 10) # Long MA: 100 to 300

print("Computing Sharpe grid for parameter optimization...")
print(f"Short MA range: {list(short_range)}")
print(f"Long MA range: {list(long_range)}")

# Compute grid (this may take a moment)
grid = compute_sharpe_grid(
    TICKER,
    str(START_DATE.date()),
    str(END_DATE.date()),
    list(short_range),
    list(long_range),
    INITIAL_CAPITAL
)

Computing Sharpe grid for parameter optimization...
Short MA range: [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80]
Long MA range: [100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260, 270, 280, 290, 300]


In [85]:
# Plot optimization heatmap
fig_opt = go.Figure(data=go.Heatmap(
    z=grid,
    x=list(long_range),
    y=list(short_range),
    colorscale='RdYlGn',
    zmid=0,
    colorbar=dict(title='Sharpe Ratio'),
    hovertemplate='Short MA: %{y} | Long MA: %{x}<br>Sharpe: %{z:.3f}<extra></extra>'
))

fig_opt.update_layout(
    title=f'{TICKER} — Parameter Optimization (Sharpe Ratio)',
    xaxis_title='Long MA (days)',
    yaxis_title='Short MA (days)',
    template='plotly_dark',
    height=500
)

fig_opt.show()

# Find best parameters
try:
    best_flat = int(np.nanargmax(grid))
    best_short = list(short_range)[best_flat // len(long_range)]
    best_long = list(long_range)[best_flat % len(long_range)]
    print(f"\n🏆 Best Parameters: Short MA = {best_short}, Long MA = {best_long}")
    print(f"   Best Sharpe Ratio: {np.nanmax(grid):.3f}")
except:
    print("\n⚠️ No valid parameter combinations found.")


🏆 Best Parameters: Short MA = 55, Long MA = 190
   Best Sharpe Ratio: 0.198


---
## 11. Key Insights

Based on the backtest results and visualizations:

1. **Trend Capture Effectiveness:** The Moving Average Crossover strategy successfully identifies and rides major price trends.

2. **Sideways Market Underperformance:** In choppy markets, the strategy may generate whipsaw trades.

3. **Drawdown Protection:** The strategy exits positions on Death Cross, providing some downside protection.

4. **Signal Lag:** MA crossovers are inherently lagging indicators.

5. **Risk-Adjusted Returns:** Compare Sharpe Ratios to evaluate if returns justify the risk taken.

6. **Trade Frequency:** Fewer trades = lower transaction costs = more capital efficiency.

7. **Regime Dependency:** Strategy performance varies significantly with market conditions.

---
## 12. Limitations

| Limitation | Impact |
|------------|--------|
| **No Transaction Costs** | Real-world commissions reduce returns |
| **No Slippage** | Assumes perfect execution at close price |
| **No Intraday Execution** | Signals assume next-day execution |
| **Past ≠ Future** | Historical results don't guarantee future performance |
| **Single Indicator** | Ignores volume, volatility, fundamentals |
| **Survivorship Bias** | Testing on successful stocks only |
| **No Risk Management** | No stop-losses or position sizing |

---
## 13. How to Run This Project

### Option 1: Interactive Web App (Streamlit)
```bash
streamlit run app.py
```
Opens at http://localhost:8501

### Option 2: Command Line
```bash
python run_strategy.py
```
Saves PNG charts and CSV data to `output/`

### Option 3: Jupyter Notebook
Open this notebook and run cells sequentially

### Requirements
```bash
pip install -r requirements.txt
```

---
## 14. Resume Bullet Points

Use these for **Quantitative Analyst, Financial Analyst, or FinTech** roles:

- **Built** a multi-strategy quantitative trading platform in Python supporting Moving Average Crossover, MA+RSI, RSI Only, and Buy & Hold strategies with full backtesting and performance evaluation.

- **Implemented** a professional-grade backtesting engine with institutional metrics (Sharpe Ratio, Maximum Drawdown, Annualized Volatility) and transaction cost modeling.

- **Designed** an interactive Streamlit dashboard with 7 tabs, Plotly visualizations, and parameter optimization heatmaps for strategy parameter selection.

- **Integrated** multi-market data from Yahoo Finance covering Nifty 50, BSE Sensex, S&P 500, Nasdaq 100, and Dow Jones indices with look-ahead bias prevention.

- **Developed** modular, reusable strategy functions using pandas and NumPy for rapid evaluation of alternative parameter configurations and strategy variants.

---
## 15. Future Enhancements

| Enhancement | Description |
|------------|-------------|
| **Walk-Forward Optimization** | Rolling in-sample/out-of-sample validation |
| **Multi-Asset Portfolio** | Backtest across basket of stocks |
| **Machine Learning Filters** | Use ML to predict signal profitability |
| **Real-Time Streaming** | Live price alerts on crossover events |
| **Paper Trading** | Connect to broker API for live execution |

---

*Built by Arnav Gupta | Not Financial Advice*